# 기본 RAG — ChromaDB 벡터 검색으로 답하기

**RAG (Retrieval-Augmented Generation)** 는 LLM 이 답하기 전에 **관련 문서를 검색(retrieve)** 해 그 내용을 근거(context)로 답을 **생성(generate)** 하는 기법이다. LLM 이 모르는 최신/전문 정보를 외부 문서로 보강해, 환각을 줄이고 출처 기반 답변을 만든다.

기본 흐름:
```
문서 → (분할/청킹) → (임베딩) → 벡터DB 저장
질문 → 벡터DB 검색 → 관련 청크 → LLM(질문+청크) → 답변
```

이 노트북에서 다루는 것:
1. 문서 로드 & 청킹 (Semantic / Recursive)
2. 임베딩 → **ChromaDB** 벡터스토어 저장 → retriever
3. **BM25 + 벡터** 하이브리드 검색 (`EnsembleRetriever`)
4. **그래프로 RAG 파이프라인** 구성 (retrieve → answer)
5. retriever 를 **도구로** 써서 LLM 이 필요할 때 검색하게 하기

> 임베딩·LLM 호출에 `OPENAI_API_KEY` 가 필요하다.

## 환경 변수 준비

`.env` 에 키를 넣고 `load_dotenv()` 로 불러온다. `USER_AGENT` 는 웹 문서 로더가 권장하는 식별자다.
```
OPENAI_API_KEY=sk-...
```

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
os.environ.setdefault("USER_AGENT", "ai-agent-study")  # WebBaseLoader 권장 식별자
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY 가 .env 에 없습니다"
print("환경변수 로드 완료")

## 1. 문서 로드 & 청킹

RAG 의 첫 단계는 소스 문서를 불러와 **검색하기 좋은 크기의 조각(chunk)** 으로 나누는 것이다. 여기서는 공개 웹 문서(LLM 에이전트 개요 글)를 소스로 사용한다.

청킹 방식 두 가지:
- **RecursiveCharacterTextSplitter**: 문자 수 기준으로 자르되 문단/문장 경계를 최대한 보존 (빠르고 무난)
- **SemanticChunker**: 임베딩 의미 유사도로 자연스러운 의미 단위에서 분할 (품질↑, 임베딩 호출 필요)

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 소스 문서: LLM 기반 에이전트 개요 (공개 웹 문서)
url = "https://lilianweng.github.io/posts/2023-06-23-agent/"
pages = WebBaseLoader(url).load()
print(f"로드된 문서 수: {len(pages)}, 본문 길이: {len(pages[0].page_content)}")

# 문자 기준 청킹
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
docs = text_splitter.split_documents(pages)
print(f"총 {len(docs)}개 청크로 분할되었습니다.")
print("첫 청크 길이:", len(docs[0].page_content))

참고: **SemanticChunker** 를 쓰면 의미 단위로 더 자연스럽게 나뉜다 (임베딩 호출이 발생).

```python
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai.embeddings import OpenAIEmbeddings

semantic_splitter = SemanticChunker(OpenAIEmbeddings())
docs = semantic_splitter.split_documents(pages)
```

## 2. 임베딩 → ChromaDB 벡터스토어

각 청크를 **임베딩**(의미를 담은 벡터)으로 바꿔 **ChromaDB**(벡터 데이터베이스)에 저장한다. 그러면 질문도 임베딩해서 **의미가 가까운 청크** 를 빠르게 찾을 수 있다.

`Chroma.from_documents(documents, embedding)` 가 청킹된 문서를 임베딩해 컬렉션으로 만든다.

In [ ]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

vectorstore = Chroma.from_documents(documents=docs, embedding=OpenAIEmbeddings())
print("벡터스토어 생성 완료")

저장된 벡터스토어에 질문을 던져 유사한 청크를 검색해본다.

In [ ]:
query = "What is task decomposition for LLM agents?"

# 직접 유사도 검색
results = vectorstore.similarity_search(query, k=1)
print(results[0].page_content[:300])

[basics 복습] 벡터스토어를 **retriever** 로 바꾸면 `invoke(query)` 로 일관된 검색 인터페이스를 쓸 수 있다.

In [ ]:
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 1})
relevant = vector_retriever.invoke(query)
print(relevant[0].page_content[:300])

## 3. 하이브리드 검색 — BM25 + 벡터 (`EnsembleRetriever`)

- **벡터 검색**: 의미적으로 비슷한 문서를 잘 찾음 (동의어/맥락에 강함)
- **BM25**: 키워드 빈도 기반 (TF-IDF 개선) — 정확한 용어/고유명사에 강함

둘을 **앙상블** 하면 서로의 약점을 보완한다. `weights` 로 비중을 조절한다.

In [ ]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

# BM25 (키워드 기반, 임베딩 불필요)
bm25_retriever = BM25Retriever.from_documents(docs)
bm25_retriever.k = 1

# 앙상블: BM25 70% + 벡터 30%
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.7, 0.3],
)

In [ ]:
q = "reflexion"
print("[Ensemble]")
for d in ensemble_retriever.invoke(q):
    print(" -", d.page_content[:100].replace(chr(10), " "))
print("\n[BM25]")
for d in bm25_retriever.invoke(q):
    print(" -", d.page_content[:100].replace(chr(10), " "))
print("\n[Vector]")
for d in vector_retriever.invoke(q):
    print(" -", d.page_content[:100].replace(chr(10), " "))

## 4. 그래프로 RAG 파이프라인 구성

검색과 답변 생성을 그래프 노드로 나눈다.
```
START → retrieve(관련 청크 검색) → answer(청크 근거로 답변) → END
```
[basics 복습] `MessagesState` 를 상속해 `context` 필드를 추가한다.

In [ ]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

class State(MessagesState):
    context: str

llm = ChatOpenAI(model="gpt-4o", temperature=0)

# 표준 RAG 프롬프트 (인라인)
RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are an assistant for question-answering tasks. "
     "Use the following retrieved context to answer the question. "
     "If you don't know the answer, just say you don't know. "
     "Use three sentences maximum and keep it concise.\n\nContext:\n{context}"),
    ("human", "{question}"),
])

In [ ]:
def retrieve(state: State):
    """질문으로 관련 청크를 검색해 context 에 담는다."""
    print("##### RETRIEVE #####")
    query = state["messages"][0].content
    results = ensemble_retriever.invoke(query)
    content = results[0].page_content
    return {"context": content, "messages": [HumanMessage(content=content)]}

def answer(state: State):
    """검색된 context 를 근거로 답변을 생성한다."""
    print("##### ANSWER #####")
    query = state["messages"][0].content
    context = state["context"]
    response = llm.invoke(RAG_PROMPT.format_messages(context=context, question=query))
    return {"messages": [response]}

In [ ]:
# [basics 복습] add_sequence 로 retrieve → answer 직선 연결
graph_builder = StateGraph(State)
graph_builder.add_sequence([retrieve, answer])
graph_builder.add_edge(START, "retrieve")
graph_builder.add_edge("answer", END)
graph = graph_builder.compile()

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    print(graph.get_graph().draw_mermaid())

In [ ]:
response = graph.invoke({"messages": "What is task decomposition?"})
for m in response["messages"]:
    m.pretty_print()

## 5. Retriever 를 도구로 — LLM 이 필요할 때 검색

위 파이프라인은 **항상** 검색한다. 하지만 인사말처럼 검색이 불필요한 질문도 있다. [basics 복습] retriever 를 **도구(tool)** 로 만들어 LLM 에 bind 하면, LLM 이 "검색이 필요한가" 를 스스로 판단한다.

```
START → chatbot ──(검색 필요?)──▶ retriever(도구) → answer → END
          └─(불필요)──────────────────────────────────────▶ END
```

In [ ]:
from langchain_core.tools.retriever import create_retriever_tool
from langgraph.prebuilt import ToolNode, tools_condition

# retriever 를 도구로 등록
retriever_tool = create_retriever_tool(
    ensemble_retriever,
    "retrieve_agent_docs",
    "Search and return information about LLM-based AI agents.",
)
tools = [retriever_tool]
llm_with_tools = llm.bind_tools(tools)

In [ ]:
def chatbot(state: State):
    # 도구 호출(tool_calls) 또는 일반 답변을 생성
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

gb = StateGraph(State)
gb.add_node("chatbot", chatbot)
gb.add_node("retriever", ToolNode(tools=tools))
gb.add_node("answer", answer)

# [basics 복습] tools_condition: 마지막 메시지에 tool_calls 있으면 'tools', 없으면 END
gb.add_conditional_edges("chatbot", tools_condition, {"tools": "retriever", END: END})
gb.add_edge(START, "chatbot")
gb.add_edge("retriever", "answer")
gb.add_edge("answer", END)
tool_rag_graph = gb.compile()

In [ ]:
try:
    display(Image(tool_rag_graph.get_graph().draw_mermaid_png()))
except Exception:
    print(tool_rag_graph.get_graph().draw_mermaid())

In [ ]:
# 검색이 필요한 질문
res = tool_rag_graph.invoke({"messages": "Explain the planning component of LLM agents."})
res["messages"][-1].pretty_print()

## 정리

- **RAG** = 검색(retrieve) + 생성(generate). 외부 문서로 LLM 답변을 근거 보강
- 파이프라인: 문서 로드 → 청킹 → 임베딩 → 벡터DB(ChromaDB) → 검색 → 답변
- **청킹**: Recursive(빠름) / Semantic(의미 단위)
- **하이브리드 검색**: 벡터(의미) + BM25(키워드) 를 `EnsembleRetriever` 로 결합
- 그래프로: `retrieve → answer`, 또는 retriever 를 **도구화** 해 LLM 이 검색 여부를 판단

이 기본 RAG 는 검색된 문서를 무조건 신뢰한다. 다음 단계에서는 검색 문서가 **정말 관련 있는지 평가**하고, 답변의 **환각 여부를 검증**하며, 부족하면 **웹에서 보강**하는 고급 RAG 로 발전시킨다.